In [56]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/vandrasembiring/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/vandrasembiring/tugas4/
print("Berhasil diunggah ke HDFS: /user/vandrasembiring/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/vandrasembiring/tugas4/transaksi_september_2026.csv


In [66]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, sum

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas4_PySpark") \
    .getOrCreate()

print("SparkSession berhasil dibuat")

SparkSession berhasil dibuat


In [58]:
path_hdfs = "hdfs://localhost:9000/user/vandrasembiring/tugas4/transaksi_september_2026.csv"

df = spark.read.csv(
    path_hdfs,
    header=True,
    inferSchema=True
)

In [42]:
print("\nSchema Data:")
df.printSchema()

print("\nJumlah Baris:")
print(df.count())

print("\n10 Baris Pertama:")
df.show(10, truncate=False)



Schema Data:
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)


Jumlah Baris:
1000

10 Baris Pertama:
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |
|ORD-3002|2026-09-26 00:00

In [71]:
jumlah_rating_kosong = df.filter(
    col("rating").isNull()
).count()

print("Jumlah rating yang kosong:", jumlah_rating_kosong)

# Mengisi rating yang kosong dengan nilai 3.0
df = df.na.fill({
    "rating": 3.0
})

jumlah_rating_setelah = df.filter(
    col("rating").isNull()
).count()

print("Jumlah rating kosong setelah ditangani:", jumlah_rating_setelah)


Jumlah rating yang kosong: 0
Jumlah rating kosong setelah ditangani: 0


In [72]:
# Membuat kolom total_pendapatan
df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

# Membuat kolom tier_transaksi
df = df.withColumn(
    "tier_transaksi",
    when(
        col("total_pendapatan") > 500000,
        "Besar"
    ).otherwise("Kecil")
)

print("\nData setelah transformasi:")
df.show(10, truncate=False)



Data setelah transformasi:
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |270000          |Kecil         |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |600000          |Besar         |
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecantikan|Semarang  |8           |60000       |E-Wallet         |3.0   |480000          |Kecil         |
|ORD-3003|2026-09-09 00:00:00|Makanan & Minuman     |Semarang  |6           |350000 

In [73]:
hasil_kategori = df.groupBy("kategori") \
    .agg(
        sum("total_pendapatan").alias("total_pendapatan")
    ) \
    .orderBy(
        col("total_pendapatan").desc()
    )

hasil_kategori.show(truncate=False)

print("Kategori dengan total pendapatan tertinggi:")
hasil_kategori.show(1, truncate=False)




hasil_kota = df.filter(
    col("tier_transaksi") == "Besar"
).groupBy("kota") \
 .count() \
 .withColumnRenamed(
     "count",
     "jumlah_transaksi_besar"
 ) \
 .orderBy(
     col("jumlah_transaksi_besar").desc()
 )

hasil_kota.show(truncate=False)

print("Kota dengan transaksi BESAR terbanyak:")
hasil_kota.show(1, truncate=False)




hasil_rating = df.groupBy("metode_pembayaran") \
    .agg(
        avg("rating").alias("rata_rata_rating")
    ) \
    .orderBy(
        col("rata_rata_rating").desc()
    )

hasil_rating.show(truncate=False)



+----------------------+----------------+
|kategori              |total_pendapatan|
+----------------------+----------------+
|Rumah Tangga          |138665000       |
|Makanan & Minuman     |131890000       |
|Kesehatan & Kecantikan|128595000       |
|Olahraga              |126650000       |
|Fashion               |124075000       |
|Elektronik            |110295000       |
+----------------------+----------------+

Kategori dengan total pendapatan tertinggi:
+------------+----------------+
|kategori    |total_pendapatan|
+------------+----------------+
|Rumah Tangga|138665000       |
+------------+----------------+
only showing top 1 row

+----------+----------------------+
|kota      |jumlah_transaksi_besar|
+----------+----------------------+
|Solo      |92                    |
|Magelang  |78                    |
|Kebumen   |78                    |
|Yogyakarta|75                    |
|Purworejo |66                    |
|Semarang  |65                    |
+----------+---------------

In [74]:
output_path = "hdfs://localhost:9000/user/vandrasembiring/tugas4/hasil"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Data berhasil disimpan ke HDFS:")
print(output_path)

Data berhasil disimpan ke HDFS:
hdfs://localhost:9000/user/vandrasembiring/tugas4/hasil


In [59]:
df_hasil = spark.read.csv(
    output_path,
    header=True,
    inferSchema=True
)

print("Jumlah baris hasil:", df_hasil.count())

print("\nData hasil dari HDFS:")
df_hasil.show(10, truncate=False)

print("\nDaftar file hasil di HDFS:")


Jumlah baris hasil: 1000

Data hasil dari HDFS:
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |270000          |Kecil         |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |600000          |Besar         |
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecantikan|Semarang  |8           |60000       |E-Wallet         |3.0   |480000          |Kecil         |
|ORD-3003|2026-09-09 00:00:00|Makanan & Minuman     |Semarang  |

In [69]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, sum

# Buat SparkSession baru
spark = SparkSession.builder \
    .appName("Tugas4_PySpark") \
    .master("local[*]") \
    .getOrCreate()

print("Spark berhasil dijalankan kembali")
print("Versi Spark:", spark.version)

# Baca ulang dataset dari HDFS
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"

df = spark.read.csv(
    path_hdfs,
    header=True,
    inferSchema=True
)

print("Data berhasil dibaca dari HDFS")
df.show(5)


Spark berhasil dijalankan kembali
Versi Spark: 3.5.9
Data berhasil dibaca dari HDFS
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
+---